# Bộ song ngữ thứ hai (Tiki): mô hình A (Gemma 4) dịch Việt sang Trung trên Kaggle, không cần GPU

Notebook này chỉ **gọi API Gemma 4 của Google AI Studio** (gói miễn phí) để dịch **tiếng Việt sang tiếng Trung giản thể** (thuộc tính sản phẩm Tiki). Không dùng GPU nên không tốn hạn mức GPU của Kaggle. Máy của anh không cần bật: Kaggle chạy nền trên máy của Kaggle, đóng trình duyệt hay tắt máy vẫn chạy.

**Cách chạy**
1. Tạo notebook mới, **File > Import notebook** chọn file này.
2. Settings (thanh bên phải): **Accelerator: None** (không GPU), **Internet: On** (bắt buộc, để gọi API của Google; tài khoản đã xác minh số điện thoại).
3. **Add-ons > Secrets**: thêm tối đa 5 secret tên `GEMINI_API_KEY`, `GEMINI_API_KEY_2`, `GEMINI_API_KEY_3`, `GEMINI_API_KEY_4`, `GEMINI_API_KEY_5` (có bao nhiêu khoá thì thêm bấy nhiêu; ô Value dán khoá của Google AI Studio, không dán vào notebook hay chat). Bấm bật (Attach) từng secret cho notebook này. Hạn mức tính theo **dự án Google** của khoá: hai khoá cùng một dự án dùng chung hạn mức nên không nhanh hơn.
   **Nhiều khoá:** mỗi yêu cầu tự chọn khoá nhanh nhất còn khoẻ; khoá lỗi hoặc quá chậm thì nghỉ và yêu cầu chuyển sang khoá khác; khoá hết hạn mức ngày hoặc bị từ chối thì loại; khoá sai bị phát hiện ngay lúc gọi thử. Số yêu cầu cùng lúc và trần mỗi phút tính theo từng khoá.
4. **Add Data**: gắn Dataset riêng tư của bộ Tiki, chứa thư mục `vao_kaggle` của `data/processed/song_ngu_2/` **bản mới nhất** (4 file, `cau_hinh_kaggle.json` phải có `huong = vi_zh`; `dich.py` phải là bản có chế độ gộp và nhiều khoá, ô thứ hai sẽ báo nếu là bản cũ).
5. Chạy thử trước: để `CHE_DO = "thu"`, **Save Version > Save & Run All (Commit)**. Vài phút là xong; xem log ô cuối.
6. Chạy thật: sửa `CHE_DO = "that"`, Save & Run All (Commit) lần nữa. Kaggle chạy nền, anh đóng tab hoặc tắt máy tuỳ ý.
7. Kết quả ở tab **Output** của phiên bản đó, thư mục `ra_gemma_tiki/`. Tải về laptop khi xong.

**Hết giờ hoặc bị ngắt thì sao:** Kaggle cho một phiên chạy nền tối đa 12 giờ; notebook tự dừng êm sau `GIO_TOI_DA` (11 giờ) để Kaggle kịp lưu đầu ra. Mỗi 500 đơn vị dịch xong được ghi ngay thành file. Muốn làm tiếp: tạo phiên bản mới, **Add Data > Notebook Output** của phiên bản trước, Save & Run All lại; script đọc phần đã có, chép sang đầu ra mới và chỉ dịch phần còn thiếu. Nếu hết hạn mức ngày của Google (14.400 yêu cầu mỗi ngày mỗi dự án), script dừng hẳn và anh chạy tiếp sau 14 giờ (giờ Việt Nam) hôm sau, cũng bằng cách gắn đầu ra cũ.

In [ ]:
# ===== Cấu hình (chỉ sửa ô này) =====
HUONG = "vi_zh"                   # chiều dịch: Việt sang Trung (bộ Tiki); không đổi
CHE_DO = "thu"                    # "thu": thử 200 đơn vị | "that": chạy cả bộ
SO_DON_VI_THU = 200
MO_HINH_API = "gemma-4-26b-a4b-it"       # nhanh (~4 giây mỗi cuộc gọi); "gemma-4-31b-it" tốt hơn nhưng đo được 35 đến 180 giây mỗi cuộc gọi
TEN_SECRETS = ["GEMINI_API_KEY_aoCw", "GEMINI_API_KEY_eVYg", "GEMINI_API_KEY_rsqw", "GEMINI_API_KEY_XIBA", "GEMINI_API_KEY_XY2w",
               "GEMINI_API_KEY", "GEMINI_API_KEY_2", "GEMINI_API_KEY_3", "GEMINI_API_KEY_4", "GEMINI_API_KEY_5"]   # tên (Label) các secret trong Kaggle Secrets; tên nào chưa tạo thì bỏ qua, đặt tên khác thì thêm vào danh sách này
API_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"
API_EXTRA = '{"reasoning_effort": "minimal"}'   # cách duy nhất Google nhận để tắt suy nghĩ của Gemma 4
GOP = 20                          # số đơn vị cùng loại gộp trong một yêu cầu
SONG_SONG_MOI_KHOA = 4            # số yêu cầu gọi cùng lúc cho mỗi khoá (tổng = số này nhân số khoá)
RPM = 20                          # trần số yêu cầu mỗi phút CHO MỖI KHOÁ (Google bản miễn phí khoảng 30 mỗi dự án)
GIO_TOI_DA = 11.0                 # giờ; Kaggle cắt phiên ở 12 giờ
NGAN_SACH_USD = 8.0               # chặn an toàn tính theo GIÁ GIẢ ĐỊNH (0,30/1,20 USD mỗi triệu token), gói miễn phí thật ra không tốn tiền; cả bộ ước ~5,4 USD theo giá giả định (2,0 là quá thấp: chạy 15/10 dừng ở 37.815/43.687)
TOI_DA_YEU_CAU = 15000

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path

def _huong(p):
    try:
        return json.loads((Path(p).parent / "cau_hinh_kaggle.json").read_text(encoding="utf-8")).get("huong")
    except Exception:
        return None

ds = [p for p in glob.glob("/kaggle/input/**/don_vi_can_dich.parquet", recursive=True) if _huong(p) == "vi_zh"]
assert ds, "Chưa gắn Dataset của bộ Tiki (don_vi_can_dich.parquet cùng cau_hinh_kaggle.json có huong = vi_zh) (Add Data)."
VAO = Path(ds[0]).parent
kich_thuoc = (VAO / "dich.py").stat().st_size
print("Dữ liệu vào:", VAO, "| dich.py", kich_thuoc, "byte")
_dich = (VAO / "dich.py").read_text(encoding="utf-8")
assert "--gop" in _dich and "KHOA_HONG" in _dich and "--huong" in _dich, "dich.py trong Dataset là bản CŨ (thiếu chế độ gộp, nhiều khoá hoặc --huong vi_zh): tải bản mới lên Dataset (Create new version)."

RA = Path("/kaggle/working/ra_gemma_tiki" if CHE_DO == "that" else "/kaggle/working/ra_gemma_tiki_thu")
RA.mkdir(parents=True, exist_ok=True)
# đầu ra phiên bản trước (Add Data > Notebook Output): thư mục ra_gemma_tiki chứa api_tuy_chon/lo_*.parquet
CU = [] if CHE_DO == "thu" else sorted({str(Path(p).parent.parent) for p in glob.glob("/kaggle/input/**/lo_*.parquet", recursive=True)
                                        if Path(p).parent.parent.name == "ra_gemma_tiki"})
print("Thư mục ra:", RA)
print("Đầu ra cũ dùng để chạy tiếp:", CU or "không có")

KHOA_ENV = []
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    _thieu = []
    for _ten in TEN_SECRETS:
        try:
            os.environ[_ten] = _us.get_secret(_ten)       # không in ra, không ghi vào file
            KHOA_ENV.append(_ten)
            print(f"Đã nạp {_ten} (độ dài {len(os.environ[_ten])} ký tự).")
        except Exception as e:
            _thieu.append(_ten)
    if _thieu:
        print("Không có secret tên:", ", ".join(_thieu), "(bỏ qua)")
except Exception as e:
    print("Không đọc được Kaggle Secrets:", type(e).__name__)
if not KHOA_ENV:
    raise SystemExit("Chưa nạp được khoá nào. Add-ons > Secrets: thêm secret GEMINI_API_KEY và bật Attach cho notebook này.")
print(f"Số khoá: {len(KHOA_ENV)} (khoá cùng dự án Google dùng chung hạn mức).")

In [ ]:
# Gọi thử Internet tới Google trước khi chạy dài (không gửi khoá, chỉ kiểm có ra được Internet)
import urllib.request
try:
    urllib.request.urlopen("https://generativelanguage.googleapis.com/", timeout=20)
except Exception as e:
    # lỗi HTTP (403/404) vẫn chứng tỏ có kết nối; chỉ lỗi mạng mới là vấn đề
    if type(e).__name__ == "HTTPError":
        print("Có kết nối tới Google (HTTP", e.code, ").")
    else:
        raise SystemExit(f"Không ra được Internet ({type(e).__name__}: {e}). Bật Settings > Internet On rồi chạy lại.")
else:
    print("Có kết nối tới Google.")

In [ ]:
thu = str(SO_DON_VI_THU if CHE_DO == "thu" else 0)
lenh = [sys.executable, str(VAO / "dich.py"), "--vao", str(VAO / "don_vi_can_dich.parquet"), "--ra", str(RA),
        "--mo_hinh", "api_tuy_chon", "--huong", HUONG, "--api_base_url", API_BASE_URL, "--api_model", MO_HINH_API, "--api_key_env", ",".join(KHOA_ENV),
        "--api_extra", API_EXTRA, "--lo", "100" if CHE_DO == "thu" else "500", "--gop", str(GOP), "--song_song", str(SONG_SONG_MOI_KHOA * len(KHOA_ENV)),
        "--rpm", str(RPM), "--toi_da_yeu_cau", str(TOI_DA_YEU_CAU), "--ngan_sach_usd", str(NGAN_SACH_USD),
        "--gio_toi_da", str(1.0 if CHE_DO == "thu" else GIO_TOI_DA), "--thu", thu] + sum([["--cu", c] for c in CU], [])
log = RA / "log_dich.log"
t0 = time.time()
p = subprocess.Popen(lenh, stdout=open(log, "w"), stderr=subprocess.STDOUT, env=dict(os.environ, PYTHONUNBUFFERED="1"))
while p.poll() is None:
    time.sleep(120)
    dong = log.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    print(f"[{(time.time() - t0) / 60:.0f} phút]", dong[-1][:230] if dong else "")
print(f"Xong sau {(time.time() - t0) / 60:.0f} phút, mã thoát {p.returncode}. 12 dòng cuối của log:")
print("\n".join(log.read_text(encoding="utf-8", errors="ignore").strip().splitlines()[-12:]))

In [ ]:
# Tóm tắt: đã dịch bao nhiêu, tốc độ, số yêu cầu, lỗi; cần chạy tiếp thì gắn đầu ra này vào phiên bản sau
import pandas as pd
tong = len(pd.read_parquet(VAO / "don_vi_can_dich.parquet", columns=["ma_don_vi"]))
so = sum(len(pd.read_parquet(f, columns=["ma_don_vi"])) for f in (RA / "api_tuy_chon").glob("lo_*.parquet"))
the = json.loads((RA / "the_api_tuy_chon.json").read_text(encoding="utf-8")) if (RA / "the_api_tuy_chon.json").exists() else {}
a_ = the.get("api", {})
print(f"Đã dịch {so:,} / {tong:,} đơn vị (chế độ {CHE_DO}); còn lại {tong - so:,}" if CHE_DO == "that" else f"Chạy thử: đã dịch {so:,} đơn vị")
for _k, _v in (a_.get("theo_khoa") or {}).items():
    print(f"  khoá {_k}: {_v}")
print("Tốc độ phiên này:", the.get("toc_do_don_vi_giay"), "đơn vị/giây | gộp:", the.get("gop"))
print("API:", {k: a_.get(k) for k in ["yeu_cau", "thu_lai", "loi_api", "luong_bi_cat", "het_quota_ngay", "het_toi_da_goi", "het_ngan_sach", "chi_phi_uoc_usd"]})
if a_.get("loi_cuoi"):
    print("Lỗi cuối:", a_["loi_cuoi"][:300])
td = the.get("toc_do_don_vi_giay")
if CHE_DO == "thu" and td:
    print(f"Ước cả bộ nếu giữ tốc độ này: {tong / td / 3600:.1f} giờ (mẫu thử trộn hai loại nên nhóm gộp nhỏ; chạy thật nhanh hơn trên mỗi đơn vị)")
if CHE_DO == "that" and tong - so > 0:
    print("CHƯA XONG: tạo phiên bản mới, Add Data > Notebook Output của phiên bản này, Save & Run All lại.")
print("\nCác file đầu ra (tab Output):")
for f in sorted(RA.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(RA.parent)}  {f.stat().st_size:,} byte")